# 06 — Performance en sous-groupes, limites, conclusion

Un modèle peut afficher une AUC correcte en moyenne et être nettement moins bon pour
certains publics. Ici je regarde si la régression logistique se comporte de la même façon
selon le sexe, l'âge et l'origine.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60)
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score
from src import features, modeling
d = joblib.load(ROOT / "models" / "baseline_logit.joblib")
pipe, X_te, y_te = d["pipeline"], d["X_te"], d["y_te"]
df = pd.read_parquet(ROOT / "data" / "processed" / "analytique.parquet")
scores = pipe.predict_proba(X_te)[:, 1]
te = df.loc[X_te.index].assign(score=scores, y=y_te.values)

In [2]:
def perf(sub):
    if sub["y"].nunique() < 2 or len(sub) < 40:
        return pd.Series({"n": len(sub), "ROC AUC": np.nan, "AP": np.nan})
    seuil = np.quantile(sub["score"], 1 - sub["y"].mean())
    return pd.Series({
        "n": len(sub),
        "prévalence": sub["y"].mean(),
        "ROC AUC": roc_auc_score(sub["y"], sub["score"]),
        "AP": average_precision_score(sub["y"], sub["score"]),
        "rappel@seuil": recall_score(sub["y"], (sub["score"] >= seuil).astype(int)),
    })

groupes = {}
groupes["Sexe"] = te.groupby("sexe").apply(perf)
te["tranche_age"] = pd.cut(te["age"], [19, 35, 50, 65, 80])
groupes["Âge"] = te.groupby("tranche_age", observed=True).apply(perf)
groupes["Origine"] = te.groupby("origine").apply(perf)
for nom, t in groupes.items():
    print(f"\n=== {nom} ==="); print(t.round(3).to_string())


=== Sexe ===
           n  prévalence  ROC AUC     AP  rappel@seuil
sexe                                                  
Femme  701.0       0.341    0.694  0.498         0.519
Homme  651.0       0.309    0.618  0.395         0.423

=== Âge ===
                 n  prévalence  ROC AUC     AP  rappel@seuil
tranche_age                                                 
(19, 35]     395.0       0.187    0.633  0.280         0.324
(35, 50]     403.0       0.323    0.621  0.412         0.438
(50, 65]     329.0       0.426    0.596  0.478         0.507
(65, 80]     225.0       0.427    0.562  0.498         0.448

=== Origine ===
                              n  prévalence  ROC AUC     AP  rappel@seuil
origine                                                                  
Asiatique non hispanique  165.0       0.230    0.638  0.373         0.395
Autre / multiracial        48.0       0.375    0.493  0.377         0.389
Blanc non hispanique      505.0       0.350    0.660  0.477         0.492


Lecture :

- **Par sexe** : performance comparable entre hommes et femmes.
- **Par âge** : le modèle est surtout discriminant chez les moins de 50 ans. Après 65 ans,
  la prévalence est si élevée que le modèle a peu de marge — l'âge sature.
- **Par origine** : plus de variabilité, en partie parce que certains sous-groupes sont
  petits dans le test. Rien qui ressemble à un effondrement systématique, mais un point à
  surveiller si le modèle devait servir à quelque chose.

In [3]:
res = pd.concat({k: v for k, v in groupes.items()}, names=["dimension", "groupe"])
res.round(3).to_csv(ROOT / "results" / "tables" / "06_performance_sous_groupes.csv")
res.round(3)

n  prévalence  ROC AUC     AP  \
dimension groupe                                                        
Sexe      Femme                     701.0       0.341    0.694  0.498   
          Homme                     651.0       0.309    0.618  0.395   
Âge       (19, 35]                  395.0       0.187    0.633  0.280   
          (35, 50]                  403.0       0.323    0.621  0.412   
          (50, 65]                  329.0       0.426    0.596  0.478   
          (65, 80]                  225.0       0.427    0.562  0.498   
Origine   Asiatique non hispanique  165.0       0.230    0.638  0.373   
          Autre / multiracial        48.0       0.375    0.493  0.377   
          Blanc non hispanique      505.0       0.350    0.660  0.477   
          Hispanique (Mexique)      201.0       0.403    0.622  0.506   
          Hispanique (autre)        144.0       0.292    0.626  0.376   
          Noir non hispanique       289.0       0.291    0.681  0.408   

                                    rappel@seuil  
dimension groupe                                  
Sexe      Femme                            0.519  
          Homme                            0.423  
Âge       (19, 35]                         0.324  
          (35, 50]                         0.438  
          (50, 65]                         0.507  
          (65, 80]                         0.448  
Origine   Asiatique non hispanique         0.395  
          Autre / multiracial              0.389  
          Blanc non hispanique             0.492  
          Hispanique (Mexique)             0.506  
          Hispanique (autre)               0.429  
          Noir non hispanique              0.429

## Ce que je retiens

**Sur la question posée.** À démographie donnée, l'alimentation et le mode de vie
*déclarés* n'expliquent presque rien du syndrome métabolique (gain de ROC AUC < 0,02).
Le risque prédictible tient à l'âge et à l'anthropométrie. C'est un résultat en creux,
mais net et reproductible sur trois cycles.

**Pourquoi ça n'est pas surprenant.** Les rappels de 24 h captent une à deux journées et
souffrent d'une sous-déclaration connue, surtout chez les personnes en surpoids. Le lien
alimentation → syndrome métabolique passe par des années d'habitudes et par
l'adiposité — deux choses qu'un instantané alimentaire mesure mal.

**Limites.**
- Sous-échantillon à jeun uniquement (~5 400 personnes) ; les non-jeûneurs pourraient
  différer.
- Pas de plan de sondage dans la modélisation (poids, strates, PSU) — volontaire, mais ça
  interdit toute lecture en termes de population.
- Cible transversale : on observe un état, pas une incidence. Un vrai modèle de risque
  demanderait un suivi longitudinal.
- Activité physique : les non-réponses ont été traitées comme « aucune activité déclarée »,
  ce qui peut sous-estimer l'activité réelle.

**Prolongements.** Analyse pondérée façon épidémiologie ; cible « pré-syndrome »
(1-2 critères) pour voir si le signal alimentaire émerge plus tôt ; passer aux données de
composition alimentaire détaillée (groupes d'aliments, score HEI) plutôt qu'aux seuls
macronutriments.